# Rally Gemma4 E4B Compare

Scores Google E4B base vs Heretic E4B.


In [ ]:
import os, platform, shutil
try:
    import torch
    print('torch=', torch.__version__, 'gpus=', torch.cuda.device_count())
except Exception as exc:
    print('torch_probe_error=', repr(exc))
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
import os, subprocess, sys, time
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        os.environ.setdefault('HF_TOKEN', token)
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', token)
        break
    except Exception:
        time.sleep(3)
packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0', 'peft>=0.19.0', 'safetensors>=0.7.0',
    'huggingface_hub[hf_transfer]>=1.5.0', 'bitsandbytes>=0.49.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
print('repo_head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
work_dir = Path('/kaggle/working/rally-e4b-heretic-compare')
base_id = os.environ.get('RALLY_BASE_MODEL_ID', 'google/gemma-4-E4B-it')
heretic_id = os.environ.get('RALLY_HERETIC_MODEL_ID', 'coder3101/gemma-4-E4B-it-heretic')
cmd = [
    sys.executable, str(REPO_DIR / 'scripts/kaggle_rally_model_compare_scorecard.py'),
    '--work-dir', str(work_dir),
    '--report-path', str(work_dir / 'rally-e4b-heretic-compare-report.json'),
    '--models', f'base:{base_id},heretic:{heretic_id}',
    '--refusal-probe-count', os.environ.get('RALLY_REFUSAL_PROBE_COUNT', '100'),
]
subprocess.check_call(cmd)


In [ ]:
import json
from pathlib import Path
report_path = Path('/kaggle/working/rally-e4b-heretic-compare/rally-e4b-heretic-compare-report.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({'ranking': report.get('ranking')}, indent=2))
